@author Nassir Mohammad

# Preliminaries

In [ ]:
import sys 
sys.path.append('../')
sys.path.append('../scripts')

import warnings
from perception_nassir import Perception

import matplotlib.pyplot as plt

import dataframe_image as dfi

import numpy as np
import pandas as pd
from scipy.io import loadmat

import seaborn as sns

from pyod.models.mcd import MCD
from sklearn.cluster import DBSCAN

from scripts.utilities import apply_classifiers

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
from sklearn.metrics import roc_auc_score
from scripts.rendering_functions import highlight_max, highlight_min

image_save_path = ''
image_save_switch = False

# ex8data2

Dataset taken from: Andrew Ng. Machine learning: Programming Exercise 8: Anomaly Detection and Recommender Systems. 2012.

## Read and explore the data

In [ ]:
data_path = r"../data/ex8data2.mat"
data = loadmat(data_path)
X2 = data['X']
X2_val = data['Xval']
y2_val = data['yval']

# show the headers
for key, val in data.items():
    print(key)
    
print("The shape of X2 is: {}".format(X2.shape))
print("The shape of X2_val is: {}".format(X2_val.shape))
print("The shape of y2 is: {}".format(y2_val.sum()))

### Standarise and view the data

In [ ]:
sc = StandardScaler()
sc.fit(X2)
X2 = sc.transform(X2)
X2_val = sc.transform(X2_val)

In [ ]:
df = pd.DataFrame(X2)
sns.pairplot(df);
plt.show()

In [ ]:
# view the validation data with labels
df2 = pd.DataFrame(np.append(X2_val, y2_val, 1))
sns.pairplot(df2, hue=11);
plt.show()

### MCD

In [ ]:
clf_mcd = MCD(random_state=42)
clf_mcd.fit(X2)
validation_labels = clf_mcd.predict(X2_val)
scores = clf_mcd.decision_function(X2_val)

print(classification_report(y2_val, validation_labels, 
                            target_names=['normal', 'abnormal'],output_dict=False))

auc_score = roc_auc_score(y2_val, scores)
print('auc score: {}'.format(auc_score))

### Perception 

In [ ]:
clf_perception = Perception()
clf_perception.fit(X2)
clf_perception.predict(X2_val)

validation_labels = clf_perception.labels_

print(classification_report(y2_val, validation_labels, 
                            target_names=['normal', 'abnormal'],output_dict=False))

auc_score = roc_auc_score(y2_val, clf_perception.scores_)
print('auc score: {}'.format(auc_score))

### DBSCAN

In [ ]:
import warnings

with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    
    clf_dbscan = DBSCAN()
    clustering = clf_dbscan.fit(X2_val) # likely not enough data to be used this way

    validation_labels = np.where(clustering.labels_ == -1, 1, 0)

    print(classification_report(y2_val, validation_labels,
                                target_names=['normal', 'abnormal'],output_dict=False))

    auc_score = roc_auc_score(y2_val, validation_labels)
    print('auc score: {}'.format(auc_score))

# Apply all models

In [ ]:
dataset_name = 'ex8data2'
classifiers = [
    'HBOS',  # to be ignored, first run in loop slower
    'HBOS',
    'IForest',
    'KNN',
    'LOF',
    'MCD',
    'OCSVM',
    'Perception',
    # 'DBSCAN',
]

df = apply_classifiers(classifiers, dataset_name,
                       predict_data=X2_val,
                       predict_labels=y2_val,
                       train_data=X2)

In [ ]:
cols = ['Precision', 'Recall', 'F1', 'AUC']
formatdict = {}
for col in cols: formatdict[col] = "{:.2f}"
formatdict.pop('Classifier', None)
formatdict['Runtime'] = "{:.5f}"
    
metrics_df_styled = df.style.hide().apply(highlight_max, 
                                                 subset=['Precision', 'Recall', 'F1', 
                                                         'AUC']).\
apply(highlight_min, subset=['Runtime']).format(formatdict)
metrics_df_styled
print(df)

In [ ]:
# title_img = 'Anomaly detection results on dataset ex8data2'    
# path = image_save_path + 'ex8data2_validation_table.png'

# dfi.export(metrics_df_styled,path)